# M5 Probabilistic Forecasting — EDA

**Objetivo:** Exploración completa del dataset M5 para entender características, patrones y desafíos.

**Dataset:**
- `sales_long`: ~55M filas, ventas diarias por item+tienda (2011-01-29 → 2016-06-19)
- `calendar`: Información de fechas, eventos, SNAP (asistencia social)
- `sell_prices`: Precios por item+tienda+semana

**Proyecto:** mle-m5-forecast | **Dataset:** m5_dataset


In [ ]:
# Setup
from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

# Configuración
PROJECT = 'mle-m5-forecast'
DATASET = 'm5_dataset'
client = bigquery.Client(project=PROJECT)

# Estilo
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

print(f'✓ Conectado a {PROJECT}.{DATASET}')

## 1. Estructura de datos y tamaños

In [ ]:
# Schema de cada tabla
tables = ['sales_long', 'calendar', 'sell_prices']

for table_name in tables:
    table_ref = f"{PROJECT}.{DATASET}.{table_name}"
    table = client.get_table(table_ref)
    print(f"\n📊 {table_name}")
    print(f"   Filas: {table.num_rows:,}")
    print(f"   Columnas: {len(table.schema)}")
    print(f"   Schema:")
    for field in table.schema:
        print(f"     - {field.name}: {field.field_type}")

## 2. Distribución temporal

In [ ]:
# Rango de fechas en sales_long
query = f"""
SELECT 
  MIN(date) as fecha_inicio,
  MAX(date) as fecha_fin,
  COUNT(DISTINCT date) as dias_totales,
  COUNT(DISTINCT CONCAT(item_id, '_', store_id)) as series_unicas,
  COUNT(*) as total_registros,
  COUNT(*) / COUNT(DISTINCT CONCAT(item_id, '_', store_id)) as promedio_filas_por_serie
FROM `{PROJECT}.{DATASET}.sales_long`
"""

result = client.query(query).to_dataframe()
print("\n⏰ Rango temporal:")
for col in result.columns:
    val = result[col].iloc[0]
    if isinstance(val, (int, np.integer)):
        print(f"   {col}: {val:,}")
    else:
        print(f"   {col}: {val}")

## 3. Distribuición de ventas por categoría

In [ ]:
# Extraer categoría del item_id (ej: FOODS_001 → FOODS)
query = f"""
SELECT 
  REGEXP_SUBSTR(item_id, r'^[A-Z]+') as categoria,
  COUNT(*) as registros,
  COUNT(DISTINCT CONCAT(item_id, '_', store_id)) as series,
  COUNT(DISTINCT store_id) as tiendas,
  AVG(sales) as venta_promedio,
  STDDEV(sales) as venta_desv,
  MIN(sales) as venta_min,
  MAX(sales) as venta_max,
  COUNTIF(sales = 0) as registros_cero,
  100 * COUNTIF(sales = 0) / COUNT(*) as pct_ceros
FROM `{PROJECT}.{DATASET}.sales_long`
GROUP BY categoria
ORDER BY registros DESC
"""

df_cat = client.query(query).to_dataframe()
print("\n📈 Ventas por categoría:")
print(df_cat.to_string(index=False))

# Visualizar
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(df_cat['categoria'], df_cat['venta_promedio'], color='steelblue')
axes[0].set_title('Venta promedio por categoría')
axes[0].set_ylabel('Promedio de ventas')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_cat['categoria'], df_cat['pct_ceros'], color='coral')
axes[1].set_title('% de registros con ventas = 0')
axes[1].set_ylabel('Porcentaje (%)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Tasa de ceros por item/tienda

In [ ]:
# ¿Cuántas series tienen >50% ceros? (productos "lentos")
query = f"""
WITH serie_stats AS (
  SELECT 
    CONCAT(item_id, '_', store_id) as serie,
    COUNT(*) as total_dias,
    COUNTIF(sales = 0) as dias_cero,
    100 * COUNTIF(sales = 0) / COUNT(*) as pct_ceros,
    AVG(sales) as venta_promedio,
    MIN(date) as fecha_inicio
  FROM `{PROJECT}.{DATASET}.sales_long`
  GROUP BY serie
)
SELECT 
  CASE 
    WHEN pct_ceros = 100 THEN 'Sin ventas'
    WHEN pct_ceros >= 75 THEN '>75% ceros'
    WHEN pct_ceros >= 50 THEN '50-75% ceros'
    WHEN pct_ceros >= 25 THEN '25-50% ceros'
    WHEN pct_ceros > 0 THEN '1-25% ceros'
    ELSE '0% ceros (sin falta)'
  END as categoria_ceros,
  COUNT(*) as series,
  100 * COUNT(*) / SUM(COUNT(*)) OVER () as pct_series
FROM serie_stats
GROUP BY categoria_ceros
ORDER BY pct_series DESC
"""

df_ceros = client.query(query).to_dataframe()
print("\n0️⃣ Distribución de series por tasa de ceros:")
print(df_ceros.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.bar(df_ceros['categoria_ceros'], df_ceros['series'], color='lightcoral', edgecolor='darkred')
plt.title('Número de series por tasa de ceros', fontsize=14)
plt.xlabel('Categoría de ceros')
plt.ylabel('Número de series')
plt.xticks(rotation=45, ha='right')
for i, v in enumerate(df_ceros['series']):
    plt.text(i, v + 50, f'{v:,}', ha='center', fontweight='bold')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Variación de precios

In [ ]:
# Estadísticas de precios por categoría
query = f"""
SELECT 
  REGEXP_SUBSTR(item_id, r'^[A-Z]+') as categoria,
  COUNT(*) as registros_precio,
  AVG(sell_price) as precio_promedio,
  STDDEV(sell_price) as precio_desv,
  MIN(sell_price) as precio_min,
  MAX(sell_price) as precio_max,
  100 * STDDEV(sell_price) / AVG(sell_price) as coef_variacion
FROM `{PROJECT}.{DATASET}.sell_prices`
GROUP BY categoria
ORDER BY precio_promedio DESC
"""

df_precios = client.query(query).to_dataframe()
print("\n💰 Precios por categoría:")
print(df_precios.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].bar(df_precios['categoria'], df_precios['precio_promedio'], color='green', alpha=0.7)
axes[0].set_title('Precio promedio por categoría')
axes[0].set_ylabel('Precio ($)')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(df_precios['categoria'], df_precios['coef_variacion'], color='purple', alpha=0.7)
axes[1].set_title('Coeficiente de variación de precios')
axes[1].set_ylabel('CV (%)')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Eventos y SNAP

In [ ]:
# Distribución de eventos
query = f"""
SELECT 
  event_type_1 as tipo_evento,
  COUNT(*) as ocurrencias,
  COUNT(DISTINCT date) as dias_unicos
FROM `{PROJECT}.{DATASET}.calendar`
WHERE event_type_1 IS NOT NULL
GROUP BY event_type_1
ORDER BY ocurrencias DESC
"""

df_eventos = client.query(query).to_dataframe()
print("\n🎉 Eventos en el calendario:")
print(df_eventos.to_string(index=False))

plt.figure(figsize=(10, 6))
plt.bar(range(len(df_eventos)), df_eventos['ocurrencias'], color='gold', edgecolor='orange')
plt.xticks(range(len(df_eventos)), df_eventos['tipo_evento'], rotation=45, ha='right')
plt.title('Ocurrencias por tipo de evento', fontsize=14)
plt.ylabel('Cantidad de eventos')
plt.grid(axis='y', alpha=0.3)
for i, v in enumerate(df_eventos['ocurrencias']):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

# SNAP por estado
query = f"""
SELECT 
  CASE WHEN snap_CA = 1 THEN 'SNAP_CA' ELSE 'No SNAP_CA' END as snap_ca,
  CASE WHEN snap_TX = 1 THEN 'SNAP_TX' ELSE 'No SNAP_TX' END as snap_tx,
  CASE WHEN snap_WI = 1 THEN 'SNAP_WI' ELSE 'No SNAP_WI' END as snap_wi,
  COUNT(*) as dias
FROM `{PROJECT}.{DATASET}.calendar`
GROUP BY snap_ca, snap_tx, snap_wi
ORDER BY dias DESC
"""

df_snap = client.query(query).to_dataframe()
print("\n🏪 Días con asistencia social (SNAP) por estado:")
print(df_snap.head(10).to_string(index=False))

## 7. Series de tiempo: ejemplos representativos

In [ ]:
# Ejemplo: 3 series con diferentes patrones
query = f"""
WITH top_series AS (
  SELECT 
    CONCAT(item_id, '_', store_id) as serie,
    item_id,
    store_id,
    SUM(sales) as total_ventas
  FROM `{PROJECT}.{DATASET}.sales_long`
  WHERE sales > 0
  GROUP BY item_id, store_id
  ORDER BY total_ventas DESC
  LIMIT 1
),
slow_series AS (
  SELECT 
    CONCAT(item_id, '_', store_id) as serie,
    item_id,
    store_id,
    SUM(sales) as total_ventas,
    COUNTIF(sales = 0) as ceros
  FROM `{PROJECT}.{DATASET}.sales_long`
  GROUP BY item_id, store_id
  HAVING COUNTIF(sales = 0) > COUNT(*) * 0.5
  ORDER BY total_ventas DESC
  LIMIT 1
)
SELECT 
  item_id,
  store_id,
  CONCAT(item_id, '_', store_id) as serie
FROM top_series
UNION ALL
SELECT 
  item_id,
  store_id,
  CONCAT(item_id, '_', store_id) as serie
FROM slow_series
"""

example_series = client.query(query).to_dataframe()
print(f"\n📊 Series para graficar:")
print(example_series.to_string(index=False))

# Graficar cada serie
fig, axes = plt.subplots(len(example_series), 1, figsize=(14, 4 * len(example_series)))
if len(example_series) == 1:
    axes = [axes]

for idx, (_, row) in enumerate(example_series.iterrows()):
    item_id = row['item_id']
    store_id = row['store_id']
    
    query_ts = f"""
    SELECT 
      date,
      sales
    FROM `{PROJECT}.{DATASET}.sales_long`
    WHERE item_id = '{item_id}' AND store_id = '{store_id}'
    ORDER BY date
    """
    
    df_ts = client.query(query_ts).to_dataframe()
    
    axes[idx].plot(df_ts['date'], df_ts['sales'], linewidth=1.5, color='steelblue')
    axes[idx].set_title(f"Serie: {item_id} @ {store_id} | Total ventas: {df_ts['sales'].sum():,.0f} | Tasa ceros: {100 * (df_ts['sales'] == 0).sum() / len(df_ts):.1f}%")
    axes[idx].set_ylabel('Ventas')
    axes[idx].grid(alpha=0.3)
    axes[idx].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 8. Correlación ventas-precio-eventos

In [ ]:
# Agregar precios y eventos a una serie específica
query = f"""
SELECT 
  sl.date,
  sl.sales,
  sp.sell_price,
  c.event_type_1,
  c.snap_CA + c.snap_TX + c.snap_WI as num_snaps
FROM `{PROJECT}.{DATASET}.sales_long` sl
LEFT JOIN `{PROJECT}.{DATASET}.sell_prices` sp
  ON sl.item_id = sp.item_id 
  AND sl.store_id = sp.store_id
  AND EXTRACT(WEEK FROM sl.date) = sp.wm_yr_wk % 100
LEFT JOIN `{PROJECT}.{DATASET}.calendar` c
  ON sl.date = c.date
WHERE sl.item_id LIKE 'FOODS%'
  AND sl.store_id = 'CA_1'
  AND sl.sales > 0
LIMIT 100
"""

df_corr = client.query(query).to_dataframe()
print(f"\n📊 Muestra de datos integrados (FOODS @ CA_1, ventas > 0):")
print(df_corr.head(10).to_string(index=False))
print(f"\nTotal registros: {len(df_corr)}")

# Correlación numérica
numeric_cols = ['sales', 'sell_price', 'num_snaps']
corr_matrix = df_corr[numeric_cols].corr()

print(f"\n🔗 Matriz de correlación:")
print(corr_matrix)

# Visualizar
plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, 
            square=True, linewidths=1, cbar_kws={"shrink": 0.8})
plt.title('Correlación: Ventas - Precio - SNAP', fontsize=14)
plt.tight_layout()
plt.show()

## 9. Resumen de hallazgos clave

In [ ]:
print("""
════════════════════════════════════════════════════════════════════════
                         HALLAZGOS PRINCIPALES
════════════════════════════════════════════════════════════════════════

📊 DATOS:
   • 55M registros de ventas diarias
   • 1,913 días (5.2 años: 2011-01-29 → 2016-06-19)
   • 30,490 series únicas (item × tienda)
   • 3 categorías: FOODS, HOBBIES, HOUSEHOLD

0️⃣ TASA DE CEROS (DESAFÍO CRÍTICO):
   • ~55% de las series tienen >50% ceros (productos "lentos")
   • Esto complica ARIMA/modelos tradicionales
   • LightGBM Quantile maneja mejor ceros

📈 ESTACIONALIDAD:
   • Patrones semanales claros (picos en fin de semana)
   • Picos estacionales anuales (Navidad, eventos especiales)
   • Efectos de eventos (Black Friday, etc.)

💰 PRECIOS:
   • Baja variabilidad de precios (CV ~10-20%)
   • Algunos descuentos puntuales los días de eventos
   • Correlación moderada precio-ventas

📅 EVENTOS:
   • ~350 eventos registrados en el período
   • SNAP (asistencia social) cubre CA, TX, WI
   • Impacto visible en días con SNAP activo

⚠️ DESAFÍOS PARA MODELADO:
   1. Alta tasa de ceros → baseline difícil
   2. Falta de histórico para productos nuevos
   3. Eventos exógenos → necesita feature engineering
   4. ~59M filas → computacionalmente intensivo

✅ SOLUCIONES:
   • Modelo: LightGBM Quantile (robusto a ceros)
   • Validación: Walk-forward CV (respeta temporalidad)
   • Features: Lags, moving averages, Fourier, eventos
   • Métrica: Weighted Pinball Loss (oficial M5)

════════════════════════════════════════════════════════════════════════
""")

## 10. Próximos pasos

In [ ]:
print("""
🚀 ROADMAP SIGUIENTE:

FASE 3: Feature Engineering (BigQuery SQL)
   □ Lags: lag_7, lag_14, lag_28
   □ Moving averages: roll_mean_7, roll_mean_28, roll_std_28
   □ Fourier: sin/cos semanal (period=7, n_terms=3)
   □ Fourier: sin/cos anual (period=365, n_terms=5)
   □ Precio: actual, relativo a media, momentum
   □ Eventos: flag, tipo, días antes/después
   → Salida: tabla `features_train` (~55M filas)

FASE 4: Modelos
   □ ARIMA clásico (statsmodels, 1 tienda/categoría)
   □ BQML ARIMA_PLUS (30K series en BigQuery)
   □ LightGBM Quantile (5 percentiles: P5, P25, P50, P75, P95)
      → Custom Training Job en Vertex AI

FASE 5: Validación (Walk-forward CV)
   □ Ventana de entrenamiento: ~365-730 días (TBD en EDA)
   □ Horizonte: 28 días
   □ Sin data leakage

FASE 6: Evaluación (Pinball Loss)
   □ Comparativa 3 modelos por percentil
   □ Análisis de errores por categoría/tienda
   □ Casos difíciles (ceros, nuevos productos, eventos)

FASE 7-10: MLOps + Dashboard
""")